In [1]:
%matplotlib inline
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import numpy.random as npr


from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

from sklearn.metrics import accuracy_score
from sklearn.metrics import roc_curve
from sklearn.metrics import roc_auc_score
from sklearn.metrics import average_precision_score
from sklearn.metrics import precision_recall_curve

# use the library eli5 for permutation importance testing
import eli5
from eli5.sklearn import PermutationImportance

# remove future warnings
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.filterwarnings(module='sklearn*', action='ignore', category=DeprecationWarning)

import os
os.environ['KMP_DUPLICATE_LIB_OK']='True'

# Set plot style
plt.style.use('fivethirtyeight')

# set figure properties
import matplotlib as mpl
mpl.rcParams['figure.dpi']= 100

In [66]:
#!pip install eli5

In [83]:
df = pd.read_csv('df_final.csv', index_col=0)

In [84]:
df.columns

Index(['rid', 'viscode', 'd1', 'd2', 'dx_bl', 'dxchange', 'age', 'ptgender',
       'pteducat', 'ptethcat', 'ptmarry', 'cdrsb_bl', 'cdrsb', 'adas13_bl',
       'adas13', 'mmse_bl', 'mmse', 'moca', 'ecogptmem', 'ecogptvisspat',
       'l_hippocampus_l', 'l_hippocampus_r', 'x_hippocampus_l',
       'x_hippocampus_r', 'l_entorhinal_l', 'l_entorhinal_r',
       'l_entorhinal_l_thick', 'l_entorhinal_r_thick', 'x_entorhinal_l',
       'x_entorhinal_r', 'x_entorhinal_l_thick', 'x_entorhinal_r_thick', 'fdg',
       'fdg_bl', 'pib', 'pib_bl', 'av45', 'av1451_hippocampus_l',
       'av1451_hippocampus_r', 'av1451_entorhinal_l', 'av1451_entorhinal_r',
       'md_hippocampus_l', 'md_hippocampus_r', 'apoe4', 'converted', 'diff_1',
       'x_hippocampus_l_bl', 'x_hippocampus_r_bl', 'x_entorhinal_l_bl',
       'x_entorhinal_r_bl', 'x_entorhinal_l_thick_bl',
       'x_entorhinal_r_thick_bl'],
      dtype='object')

In [85]:
#df.head()

In [86]:
# drop patients with baseline conversion to AD
df = df.loc[df.dx_bl != 'AD']
# one hot encode gender
features = pd.get_dummies(df, columns=['ptgender', 'dx_bl'])

In [87]:
features.columns

Index(['rid', 'viscode', 'd1', 'd2', 'dxchange', 'age', 'pteducat', 'ptethcat',
       'ptmarry', 'cdrsb_bl', 'cdrsb', 'adas13_bl', 'adas13', 'mmse_bl',
       'mmse', 'moca', 'ecogptmem', 'ecogptvisspat', 'l_hippocampus_l',
       'l_hippocampus_r', 'x_hippocampus_l', 'x_hippocampus_r',
       'l_entorhinal_l', 'l_entorhinal_r', 'l_entorhinal_l_thick',
       'l_entorhinal_r_thick', 'x_entorhinal_l', 'x_entorhinal_r',
       'x_entorhinal_l_thick', 'x_entorhinal_r_thick', 'fdg', 'fdg_bl', 'pib',
       'pib_bl', 'av45', 'av1451_hippocampus_l', 'av1451_hippocampus_r',
       'av1451_entorhinal_l', 'av1451_entorhinal_r', 'md_hippocampus_l',
       'md_hippocampus_r', 'apoe4', 'converted', 'diff_1',
       'x_hippocampus_l_bl', 'x_hippocampus_r_bl', 'x_entorhinal_l_bl',
       'x_entorhinal_r_bl', 'x_entorhinal_l_thick_bl',
       'x_entorhinal_r_thick_bl', 'ptgender_Female', 'ptgender_Male',
       'dx_bl_CN', 'dx_bl_LMCI'],
      dtype='object')

In [88]:
core_features = ['rid', 'viscode', 'age', 'pteducat', 
                 'cdrsb', 'adas13', 'mmse', 
                'x_hippocampus_l', 'x_hippocampus_r', 'x_entorhinal_l', 
                'x_entorhinal_r', 'x_entorhinal_l_thick', 'x_entorhinal_r_thick',
                 'diff_1', 'apoe4', 'ptgender_Female', 'ptgender_Male', 'dx_bl_CN', 'dx_bl_LMCI', 'converted']

In [89]:
features.converted.value_counts()

0    2822
1    1631
Name: converted, dtype: int64

In [90]:
# extract non converter datapoints
df_non = features.loc[features.converted==0]
# extract all converter datapoints
df_converted = features.loc[features.converted==1]
# concatenate
df_core = pd.concat([df_non, df_converted])
# filter to core feature set
df_core = df_core.loc[:,core_features]
# drop NaN rows
df_core.dropna(inplace=True)

In [91]:
df_core.shape

(3390, 20)

In [92]:
df_core.converted.value_counts()

0    2072
1    1318
Name: converted, dtype: int64

In [93]:
df_core.columns

Index(['rid', 'viscode', 'age', 'pteducat', 'cdrsb', 'adas13', 'mmse',
       'x_hippocampus_l', 'x_hippocampus_r', 'x_entorhinal_l',
       'x_entorhinal_r', 'x_entorhinal_l_thick', 'x_entorhinal_r_thick',
       'diff_1', 'apoe4', 'ptgender_Female', 'ptgender_Male', 'dx_bl_CN',
       'dx_bl_LMCI', 'converted'],
      dtype='object')

In [15]:
df_core.to_csv('df_model.csv')

In [94]:
df_core.columns

Index(['rid', 'viscode', 'age', 'pteducat', 'cdrsb', 'adas13', 'mmse',
       'x_hippocampus_l', 'x_hippocampus_r', 'x_entorhinal_l',
       'x_entorhinal_r', 'x_entorhinal_l_thick', 'x_entorhinal_r_thick',
       'diff_1', 'apoe4', 'ptgender_Female', 'ptgender_Male', 'dx_bl_CN',
       'dx_bl_LMCI', 'converted'],
      dtype='object')

In [95]:
df3 = df_core.drop(['rid', 'viscode', 'mmse',  'x_entorhinal_r', 'apoe4', 'ptgender_Female', 'ptgender_Male', 'dx_bl_CN',
       'dx_bl_LMCI'], axis=1)

In [96]:
df3

,age,pteducat,cdrsb,adas13,x_hippocampus_l,x_hippocampus_r,x_entorhinal_l,x_entorhinal_l_thick,x_entorhinal_r_thick,diff_1,converted
10,76.3,18,0.0,9.33,3573.0,3976.0,1863.0,3.061,3.655,-9.33,0
11,76.3,18,0.0,4.67,3614.0,3792.0,1744.0,3.110,3.638,-9.33,0
12,76.3,18,0.0,9.33,3408.0,3806.0,1510.0,2.995,3.259,-9.33,0
14,76.3,18,2.0,5.33,3619.0,3877.0,1598.0,2.892,3.353,-9.33,0
15,76.3,18,0.5,9.00,3302.0,3755.0,1806.0,2.972,3.281,-9.33,0
...,...,...,...,...,...,...,...,...,...,...,...
5250,80.4,20,3.5,27.00,1784.0,2033.0,806.0,2.006,2.376,-18.00,1
5251,80.4,20,4.5,34.00,1591.0,1755.0,613.0,1.680,2.588,-18.00,1
5252,80.4,20,5.5,30.33,1626.0,1938.0,837.0,1.986,2.749,-18.00,1
5253,80.4,20,4.5,37.33,1615.0,1661.0,794.0,1.738,2.261,-18.00,1


In [101]:
df3.to_csv('df_model2.csv')

In [102]:
df4 = pd.read_csv('df_model2.csv')

In [107]:
df4 = pd.read_csv('df_model2.csv', index_col=[0])

In [108]:
df4

,age,pteducat,cdrsb,adas13,x_hippocampus_l,x_hippocampus_r,x_entorhinal_l,x_entorhinal_l_thick,x_entorhinal_r_thick,diff_1,converted
10,76.3,18,0.0,9.33,3573.0,3976.0,1863.0,3.061,3.655,-9.33,0
11,76.3,18,0.0,4.67,3614.0,3792.0,1744.0,3.110,3.638,-9.33,0
12,76.3,18,0.0,9.33,3408.0,3806.0,1510.0,2.995,3.259,-9.33,0
14,76.3,18,2.0,5.33,3619.0,3877.0,1598.0,2.892,3.353,-9.33,0
15,76.3,18,0.5,9.00,3302.0,3755.0,1806.0,2.972,3.281,-9.33,0
...,...,...,...,...,...,...,...,...,...,...,...
5250,80.4,20,3.5,27.00,1784.0,2033.0,806.0,2.006,2.376,-18.00,1
5251,80.4,20,4.5,34.00,1591.0,1755.0,613.0,1.680,2.588,-18.00,1
5252,80.4,20,5.5,30.33,1626.0,1938.0,837.0,1.986,2.749,-18.00,1
5253,80.4,20,4.5,37.33,1615.0,1661.0,794.0,1.738,2.261,-18.00,1


In [104]:
#Import single models
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.svm import SVR, SVC
from sklearn.neighbors import KNeighborsRegressor,KNeighborsClassifier

#set seed
rand_seed = 234
np.random.seed = rand_seed

#Classification models
log_cf = LogisticRegression(solver='lbfgs', random_state=rand_seed)
svc_cf = SVC(gamma='scale', random_state=rand_seed)
knn_cf = KNeighborsClassifier()

classification_models = [log_cf, svc_cf, knn_cf]

#Regression models
linear_reg = LinearRegression()
svr_reg = SVR(gamma='scale')
knn_reg = KNeighborsRegressor()

regression_models = [linear_reg, svr_reg, knn_reg]

In [105]:
#Define a function to standardize the data set
def standardize_data(df):
    scaler = RobustScaler()
    data = scaler.fit_transform(df)
    return data

#Create a function to split our data into train and validation set for both task

def get_split_data(features, target_name=None):
    ## Get the target column
    target = features[target_name]
    ## Drop the target from the data
    temp_data = features.drop(target_name, axis=1)
    temp_data = standardize_data(temp_data)
    
    #split data
    X_train, X_val, y_train, y_val = train_test_split(temp_data, target, test_size=0.1)
    return (X_train, X_val, y_train, y_val)
    
def get_mae(pred, true_value):
    return mean_absolute_error(true_value, pred)


def get_acc(pred, true_value):
    return accuracy_score(true_value, pred) * 100

# A Function to train and cross validate a model
def model_train(model, features=None, target_name=None, nfolds = 10, task = 'class'):
    ## Get the target column
    target = features[target_name]
    ## Drop the target from the data
    temp_data = features.drop(target_name, axis=1)
    temp_data = standardize_data(temp_data)
    
    if task == 'reg':
        score = -1 * (cross_val_score(model, temp_data, target, cv=nfolds, scoring='neg_mean_absolute_error'))
        print("Mean Absolute Error of {} is {}".format(model.__class__.__name__, round(score[0], 4)))
        print("-------------------------------------")

    else:
        score = cross_val_score(model, temp_data, target, cv=nfolds, scoring='accuracy')
        print("Accuracy of {} is {} %".format(model.__class__.__name__, round(score[0] * 100)))
        print("-------------------------------------")

In [110]:
#Metric calculations
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import mean_absolute_error, accuracy_score


from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split

In [111]:
#Classification 
for model in classification_models:
    model_train(model, features=df4, target_name='converted')

Accuracy of LogisticRegression is 78 %
-------------------------------------
Accuracy of SVC is 75 %
-------------------------------------
Accuracy of KNeighborsClassifier is 75 %
-------------------------------------


In [112]:
#get the data sets
X_train, X_val, y_train, y_val = get_split_data(df4, target_name='converted')

#fit base models
linear_reg.fit(X_train, y_train)
knn_reg.fit(X_train, y_train)
svr_reg.fit(X_train, y_train)

#make predictions with trained models
pred1 = linear_reg.predict(X_val)
pred2 = knn_reg.predict(X_val)
pred3 = svr_reg.predict(X_val)

#Take average as final prediction
avgpred = (pred1 + pred2 + pred3) / 3

In [113]:
print("Linear Regression Model")
print(get_mae(pred1, y_val))
print("KNN Regression Model")
print(get_mae(pred2, y_val))
print("SVR Regression Model")
print(get_mae(pred3, y_val))
print("Average Model")
print(get_mae(avgpred, y_val))

Linear Regression Model
0.30673227142550724
KNN Regression Model
0.12743362831858407
SVR Regression Model
0.2068823348556133
Average Model
0.2046539435311087


In [114]:
#fit base models
linear_reg.fit(X_train, y_train)
knn_reg.fit(X_train, y_train)
svr_reg.fit(X_train, y_train)

#make predictions with trained models
pred1 = linear_reg.predict(X_val)
pred2 = knn_reg.predict(X_val)
pred3 = svr_reg.predict(X_val)

#Take average as final prediction
w_avgpred = (0.5 * pred1 + 0.25 * pred2 + 0.25* pred3)

print("Linear Regression Model")
print(get_mae(pred1, y_val))
print("KNN Regression Model")
print(get_mae(pred2, y_val))
print("SVR Regression Model")
print(get_mae(pred3, y_val))
print("Weighted Average Model")
print(get_mae(w_avgpred, y_val))

Linear Regression Model
0.30673227142550724
KNN Regression Model
0.12743362831858407
SVR Regression Model
0.2068823348556133
Weighted Average Model
0.22948420071384873


In [116]:
from statistics import mode
#get the data sets
X_train, X_val, y_train, y_val = get_split_data(df4, target_name='converted')

#fit single models
log_cf.fit(X_train, y_train)
knn_cf.fit(X_train, y_train)
svc_cf.fit(X_train, y_train)

#make predictions with trained models
pred1 = log_cf.predict(X_val)
pred2 = knn_cf.predict(X_val)
pred3 = svc_cf.predict(X_val)

#Take max voting as final prediction
maxpred = []

for i in range(0, len(X_val)):
    #calculate the mode and append to maxpred vector
    maxpred.append(mode([pred1[i], pred2[i], pred3[i]]))
    
    
print("Logistic Regression Model")
print(get_acc(pred1, y_val))
print("KNN Classifier Model")
print(get_acc(pred2, y_val))
print("SVR Classifier Model")
print(get_acc(pred3, y_val))

print("Max Voting Model")
print(get_acc(np.array(maxpred), y_val))

Logistic Regression Model
79.9410029498525
KNN Classifier Model
88.49557522123894
SVR Classifier Model
84.36578171091446
Max Voting Model
83.77581120943954


In [117]:
#Import the module
from sklearn.ensemble import VotingClassifier

#Pass the classifiers as a list of tuples with model names and the models themselves
max_model = VotingClassifier(estimators=[('logistic_reg', log_cf), ('KNN Classifier', knn_cf), ("SVC", svc_cf)], voting='hard')
max_model.fit(X_train, y_train)

print("Max Voting in sklearn")
print(get_acc(max_model.predict(X_val), y_val))

Max Voting in sklearn
83.77581120943954


In [118]:
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, BaggingRegressor
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, BaggingClassifier
from sklearn.ensemble import GradientBoostingRegressor, AdaBoostRegressor
#import xgboost as xgb


#bagging algorithms for regression
rand_forest_reg = RandomForestRegressor(n_estimators=100, random_state=rand_seed)
extra_tree_reg = ExtraTreesRegressor(n_estimators=100,random_state=rand_seed)
#We use linear regressor as our base model for bagging
bagging_meta_reg = BaggingRegressor(svr_reg, n_estimators=100, random_state=rand_seed)

#bagging algorithms for classification
rand_forest_cf = RandomForestClassifier(n_estimators=100, random_state=rand_seed)
extra_tree_cf = ExtraTreesClassifier(n_estimators=100, random_state=rand_seed)
#We use svc as our base model for bagging
bagging_meta_cf = BaggingClassifier(svc_cf, n_estimators=10, random_state=rand_seed)

In [119]:
#get data for regression task
X_train, X_val, y_train, y_val = get_split_data(df4, target_name='converted')

#Train and fit these models
rand_forest_reg.fit(X_train, y_train)
extra_tree_reg.fit(X_train, y_train)
bagging_meta_reg.fit(X_train, y_train)

#check their performance
print("MAE of Random Forest is : ", get_mae(rand_forest_reg.predict(X_val), y_val))
print("MAE of Extra Trees is : ", get_mae(extra_tree_reg.predict(X_val), y_val))
print("MAE of Bagging estimator is : ", get_mae(bagging_meta_reg.predict(X_val), y_val))

MAE of Random Forest is :  0.18404129793510327
MAE of Extra Trees is :  0.17404129793510326
MAE of Bagging estimator is :  0.18958175178284375


In [120]:
#get data for classification task
X_train, X_val, y_train, y_val = get_split_data(df4, target_name='converted')

#Train and fit these models
rand_forest_cf.fit(X_train, y_train)
extra_tree_cf.fit(X_train, y_train)
bagging_meta_cf.fit(X_train, y_train)

#check their performance
print("ACC of Random Forest is : ", get_acc(rand_forest_cf.predict(X_val), y_val))
print("ACC of Extra Trees is : ", get_acc(extra_tree_cf.predict(X_val), y_val))
print("ACC of Bagging estimator is : ", get_acc(bagging_meta_cf.predict(X_val), y_val))

ACC of Random Forest is :  87.90560471976401
ACC of Extra Trees is :  91.1504424778761
ACC of Bagging estimator is :  84.66076696165192


In [122]:
def stackingModel(base_models, meta_model, features, target, nfolds=10):
    #Split data into folds
    kfold = KFold(n_splits=nfolds, shuffle=True, random_state=rand_seed)
    #initialize arrays to hold predictions
    test_predictions = np.zeros((features.shape[0], len(base_models)))
    train_predictions = np.zeros((features.shape[0], len(base_models)))
    
    # Train base models
    for i, model in enumerate(base_models):
        for train_index, test_index in kfold.split(features, target):
            #Fit train data on the model
            model.fit(np.array(features)[train_index], np.array(target)[train_index])
            
            #Make prediction on the holdout data
            y_pred = model.predict(np.array(features)[test_index])
            #make predictions on train data
            t_pred = model.predict(np.array(features)[train_index])
            
            #Append the prediction to out of folds
            test_predictions[test_index, i] = y_pred
            #Append predictions to train predictions
            train_predictions[train_index, i] = t_pred


    # Now train the meta-model using the train predictions as new feature
    meta_model.fit(train_predictions, target)
    #Make fianl predictions on the average of out of fold predictions
    final_preds = meta_model.predict(np.mean([test_predictions], axis=0))
    
    return final_preds

In [124]:
#get data for classification task
target = df4['converted']
data = df4.drop('converted', axis=1)
data = standardize_data(data)

#first level learners
base_learners = [log_cf, svc_cf, knn_cf]
#meta learner
meta_ln = svc_cf

pred = stackingModel(base_learners, meta_ln, data, target)

#check performance
print("ACC of Stacking Model is : ", get_acc(pred ,target))

ACC of Stacking Model is :  90.53097345132744
